# 🏦 Bank Term Deposit Prediction — ML Pipeline
---
**Dataset:** Bank Marketing (bank-full.csv) — 45,211 rows
**Goal:** Predict if a customer will subscribe to a Term Deposit (`y = yes / no`)
**ML Type:** Supervised Binary Classification

**Pipeline followed:**
1. Data Importing
2. EDA & Correlation Analysis
3. Variable Selection (Forward Selection, Backward Elimination, RFE)
4. Baseline Model — Simple Logistic Regression
5. Underfitting / Overfitting Check
6. Advanced Models — Decision Tree, Random Forest, XGBoost

> **Author:** Data Science Project
> **Documents Referenced:** ML_PRO_DES.docx | PRO_DATA_DICTONARY.docx


## 📦 Step 1A. Import Libraries

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import RFE

# Statsmodels — for Forward Selection / Backward Elimination (p-value based)
import statsmodels.api as sm

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report)

print("✅ Libraries imported successfully")


## 📂 Step 1B. Data Importing

In [ ]:
# The file is semicolon-delimited despite the .xls extension
df = pd.read_csv('bank-full_csv.xls', sep=';')

print(f"✅ Data Loaded Successfully!")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
df.head()


In [ ]:
df.info()


## 🔍 Step 2. Exploratory Data Analysis (EDA) & Correlation Analysis

In [ ]:
# Data Types & Missing Values
print("─── Data Types ───────────────────────────────")
print(df.dtypes)
print("\n─── Missing Values ───────────────────────────")
print(df.isnull().sum())
print("\n─── Duplicate Rows ───────────────────────────")
print(f"Duplicates: {df.duplicated().sum()}")


In [ ]:
# Statistical Summary
df.describe(include='all').T


In [ ]:
# Target Variable Distribution
print("─── Target Variable (y) Distribution ────────")
vc = df['y'].value_counts()
print(vc)
print(f"\nClass Imbalance Ratio: {round(vc['no'] / vc['yes'], 2)} : 1")
print("⚠️  Dataset is imbalanced — SMOTE will be applied before advanced modeling (Step 6).")


In [ ]:
# ── EDA Dashboard ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Bank Term Deposit — EDA Dashboard', fontsize=16, fontweight='bold')

colors = ['#e74c3c', '#2ecc71']

# 1. Target Distribution
df['y'].value_counts().plot(kind='bar', ax=axes[0,0], color=colors, edgecolor='black')
axes[0,0].set_title('Target Variable Distribution')
axes[0,0].set_xlabel('Subscribed Term Deposit?')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=0)

# 2. Age Distribution
sns.histplot(df['age'], bins=30, kde=True, ax=axes[0,1], color='#3498db')
axes[0,1].set_title('Age Distribution')

# 3. Balance Distribution (clipped for readability)
sns.histplot(df['balance'].clip(-2000, 20000), bins=40, kde=True, ax=axes[0,2], color='#9b59b6')
axes[0,2].set_title('Account Balance Distribution (clipped)')

# 4. Job vs Subscription rate
job_rate = df.groupby('job')['y'].apply(lambda x: (x == 'yes').mean()).sort_values()
job_rate.plot(kind='barh', ax=axes[1,0], color='#f39c12')
axes[1,0].set_title('Subscription Rate by Job')

# 5. Education vs Subscription rate
edu_rate = df.groupby('education')['y'].apply(lambda x: (x == 'yes').mean()).sort_values()
edu_rate.plot(kind='barh', ax=axes[1,1], color='#16a085')
axes[1,1].set_title('Subscription Rate by Education')

# 6. Month vs Subscription rate
month_rate = df.groupby('month')['y'].apply(lambda x: (x == 'yes').mean())
month_rate.plot(kind='bar', ax=axes[1,2], color='#c0392b')
axes[1,2].set_title('Subscription Rate by Month')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# ── Correlation Analysis ──────────────────────────────────────────────────
# Encode a temporary copy purely for correlation inspection (not used for modeling)
corr_df = df.copy()
corr_df['y'] = (corr_df['y'] == 'yes').astype(int)

cat_cols_temp = corr_df.select_dtypes(include='object').columns.tolist()
for col in cat_cols_temp:
    corr_df[col] = LabelEncoder().fit_transform(corr_df[col])

plt.figure(figsize=(14, 10))
corr_matrix = corr_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Heatmap — All Features (incl. Target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("─── Correlation of each feature with target (y) ───")
print(corr_matrix['y'].sort_values(ascending=False))


**EDA / Correlation takeaways**
- `duration` has the strongest correlation with `y`, but it is a **leakage feature** (only known after the call happens) → dropped in feature engineering below.
- `pdays` and `previous` are correlated with each other (customers contacted before tend to have both populated) → handled via the `was_contacted_before` flag.
- No feature shows extreme multicollinearity (|r| > 0.85) with another predictor, so no features are dropped purely on correlation grounds; a formal statistical selection is done in Step 3.


## ⚙️ Step 2B. Feature Engineering

**Key decisions based on the Data Dictionary:**
- **Drop `duration`** — Call duration is only known *after* the call ends → causes **data leakage**
- **`pdays = -1`** means "never contacted before" → create binary flag `was_contacted_before`
- **Encode all categorical columns** using Label Encoding


In [ ]:
data = df.copy()

# 1. Drop duration (data leakage)
data.drop(columns=['duration'], inplace=True)
print("✂️  Dropped 'duration' — Data Leakage Risk")

# 2. Engineer pdays feature
data['was_contacted_before'] = (data['pdays'] != -1).astype(int)
data['pdays'] = data['pdays'].replace(-1, 0)
print("✅  Created 'was_contacted_before' binary feature")

# 3. Encode target variable
data['y'] = (data['y'] == 'yes').astype(int)
print("✅  Target 'y' encoded: yes=1, no=0")

# 4. Label encode all remaining categorical columns
cat_cols = data.select_dtypes(include='object').columns.tolist()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le
print(f"✅  Label-encoded columns: {cat_cols}")

data.head()


## 🧮 Step 3. Variable Selection Process
Three complementary techniques are used to decide which features actually earn a place in the model:

1. **Backward Elimination** — start with all features, repeatedly drop the one with the highest p-value (statsmodels Logit) until everything left is significant (p < 0.05)
2. **Forward Selection** — start with no features, repeatedly add the one that improves AIC the most, until no further improvement
3. **Recursive Feature Elimination (RFE)** — let scikit-learn rank features by importance to a Logistic Regression estimator and keep the top N


In [ ]:
X_full = data.drop('y', axis=1)
y_full = data['y']
feature_names = X_full.columns.tolist()


In [ ]:
# ── 3A. Backward Elimination (statsmodels, p-value based) ────────────────
def backward_elimination(X, y, sl=0.05):
    features = list(X.columns)
    while len(features) > 0:
        X_const = sm.add_constant(X[features])
        model = sm.Logit(y, X_const).fit(disp=0)
        p_values = model.pvalues.iloc[1:]  # exclude intercept
        max_p = p_values.max()
        if max_p > sl:
            worst_feature = p_values.idxmax()
            features.remove(worst_feature)
        else:
            break
    return features, model

be_features, be_model = backward_elimination(X_full, y_full)
print(f"✅ Backward Elimination kept {len(be_features)} features:")
print(be_features)
print("\n" + be_model.summary().as_text())


In [ ]:
# ── 3B. Forward Selection (AIC based) ─────────────────────────────────────
def forward_selection(X, y):
    remaining = list(X.columns)
    selected = []
    current_aic = np.inf

    while remaining:
        aic_candidates = []
        for feature in remaining:
            trial_features = selected + [feature]
            X_const = sm.add_constant(X[trial_features])
            model = sm.Logit(y, X_const).fit(disp=0)
            aic_candidates.append((model.aic, feature))

        aic_candidates.sort()
        best_aic, best_feature = aic_candidates[0]

        if best_aic < current_aic:
            selected.append(best_feature)
            remaining.remove(best_feature)
            current_aic = best_aic
        else:
            break
    return selected

fs_features = forward_selection(X_full, y_full)
print(f"✅ Forward Selection kept {len(fs_features)} features:")
print(fs_features)


In [ ]:
# ── 3C. Recursive Feature Elimination (RFE) ───────────────────────────────
n_features_to_select = 10

rfe_estimator = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=n_features_to_select)
rfe.fit(X_full, y_full)

rfe_features = X_full.columns[rfe.support_].tolist()
print(f"✅ RFE kept top {n_features_to_select} features:")
print(rfe_features)

rfe_ranking = pd.Series(rfe.ranking_, index=X_full.columns).sort_values()
print("\n─── RFE Feature Ranking (1 = selected) ───")
print(rfe_ranking)


In [ ]:
# ── Compare the three selection methods & pick the final feature set ─────
common_features = sorted(set(be_features) & set(fs_features) & set(rfe_features))
print(f"Backward Elimination : {sorted(be_features)}")
print(f"Forward Selection    : {sorted(fs_features)}")
print(f"RFE                  : {sorted(rfe_features)}")
print(f"\n✅ Features agreed upon by all 3 methods ({len(common_features)}): {common_features}")

# Final feature set: union of Backward Elimination + RFE (both are the most
# statistically / model-driven), used going forward for the baseline model.
selected_features = sorted(set(be_features) | set(rfe_features))
print(f"\n📌 Final selected feature set used for modeling ({len(selected_features)}): {selected_features}")


## 📉 Step 4. Baseline Model — Simple Logistic Regression
Since the target `y` is binary (subscribed / not subscribed), **Logistic Regression** is the
appropriate simple/baseline model (a Linear Regression model is not applicable here because
the target is categorical, not continuous). We train it only on the features selected in Step 3.


In [ ]:
X = data[selected_features]
y_target = data['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_target, test_size=0.2, random_state=42, stratify=y_target
)

print(f"Train set : {X_train.shape[0]:,} rows")
print(f"Test  set : {X_test.shape[0]:,} rows")
print(f"Features  : {X_train.shape[1]} -> {selected_features}")


In [ ]:
# Scale numeric features (important for Logistic Regression convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

baseline_lr = LogisticRegression(max_iter=1000, random_state=42)
baseline_lr.fit(X_train_scaled, y_train)

y_train_pred = baseline_lr.predict(X_train_scaled)
y_test_pred  = baseline_lr.predict(X_test_scaled)
y_test_proba = baseline_lr.predict_proba(X_test_scaled)[:, 1]

print("─── Baseline Logistic Regression Performance ───")
print(f"Train Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Test  Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test  Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Test  Recall   : {recall_score(y_test, y_test_pred):.4f}")
print(f"Test  F1-score : {f1_score(y_test, y_test_pred):.4f}")
print(f"Test  ROC-AUC  : {roc_auc_score(y_test, y_test_proba):.4f}")


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No (Pred)', 'Yes (Pred)'],
            yticklabels=['No (Actual)', 'Yes (Actual)'])
plt.title('Baseline Logistic Regression — Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()


## 🧪 Step 5. Checking for Underfitting / Overfitting
We compare train vs. test performance, run k-fold cross-validation, and plot a learning
curve for the baseline Logistic Regression model.

- **Underfitting**: both train and test scores are low and close together
- **Overfitting**: train score is much higher than test score
- **Good fit**: train and test scores are both reasonably high and close together


In [ ]:
train_acc = accuracy_score(y_train, y_train_pred)
test_acc  = accuracy_score(y_test, y_test_pred)
gap = train_acc - test_acc

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test  Accuracy : {test_acc:.4f}")
print(f"Gap (Train-Test): {gap:.4f}")

if gap > 0.05:
    print("⚠️  Possible OVERFITTING — train accuracy notably higher than test accuracy")
elif train_acc < 0.75 and test_acc < 0.75:
    print("⚠️  Possible UNDERFITTING — both train and test accuracy are low")
else:
    print("✅ Model shows a reasonable fit — train/test scores are close")


In [ ]:
# 5-Fold Cross-Validation on the training set
cv_scores = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"Cross-Validation ROC-AUC scores : {np.round(cv_scores, 4)}")
print(f"Mean CV ROC-AUC                 : {cv_scores.mean():.4f}")
print(f"Std  CV ROC-AUC                 : {cv_scores.std():.4f}")
print("A low standard deviation across folds indicates the model generalizes consistently.")


In [ ]:
# Learning Curve — train size vs. score
train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_scaled, y_train, cv=5, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#2980b9', label='Training score')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#e74c3c', label='Validation score')
plt.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                  train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15, color='#2980b9')
plt.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                  val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15, color='#e74c3c')
plt.title('Learning Curve — Baseline Logistic Regression')
plt.xlabel('Training Set Size')
plt.ylabel('ROC-AUC Score')
plt.legend(loc='best')
plt.tight_layout()
plt.show()


## 🤖 Step 6. Advanced Models — Decision Tree, Random Forest & XGBoost
The baseline Logistic Regression gives us a benchmark. Now we train tree-based / ensemble
models, which typically capture non-linear relationships better. Because the training data
is imbalanced, **SMOTE** is applied first (test set is left untouched, to avoid leakage).


In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE → {y_train.value_counts().to_dict()}")
print(f"After  SMOTE → {pd.Series(y_train_sm).value_counts().to_dict()}")

# Re-scale after resampling (tree models don't need scaling, but Logistic Regression
# is kept in the comparison below, so we scale consistently)
scaler_adv = StandardScaler()
X_train_sm_scaled = scaler_adv.fit_transform(X_train_sm)
X_test_sm_scaled  = scaler_adv.transform(X_test)
print("✅ SMOTE applied and features scaled")


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost'            : XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                          eval_metric='logloss', random_state=42, n_jobs=-1),
}

results = {}
print(f"{'Model':<22} {'Train Acc':>10} {'Test Acc':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'ROC-AUC':>10}")
print("-" * 90)

for name, model in models.items():
    model.fit(X_train_sm_scaled, y_train_sm)
    y_tr_pred = model.predict(X_train_sm_scaled)
    y_pred    = model.predict(X_test_sm_scaled)
    y_proba   = model.predict_proba(X_test_sm_scaled)[:, 1]

    results[name] = {
        'model': model,
        'train_acc': accuracy_score(y_train_sm, y_tr_pred),
        'test_acc' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'f1'       : f1_score(y_test, y_pred),
        'roc_auc'  : roc_auc_score(y_test, y_proba),
        'y_pred'   : y_pred,
        'y_proba'  : y_proba,
    }
    r = results[name]
    print(f"{name:<22} {r['train_acc']:>10.4f} {r['test_acc']:>10.4f} {r['precision']:>10.4f} "
          f"{r['recall']:>10.4f} {r['f1']:>10.4f} {r['roc_auc']:>10.4f}")

best_model_name = max(results, key=lambda k: results[k]['roc_auc'])
best = results[best_model_name]
print(f"\n🏆 Best Model: {best_model_name} (ROC-AUC = {best['roc_auc']:.4f})")


In [ ]:
# ── Overfitting check for every advanced model ────────────────────────────
print(f"{'Model':<22} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>10}  Diagnosis")
print("-" * 75)
for name, r in results.items():
    gap = r['train_acc'] - r['test_acc']
    if gap > 0.08:
        diag = "⚠️ Overfitting"
    elif r['train_acc'] < 0.75 and r['test_acc'] < 0.75:
        diag = "⚠️ Underfitting"
    else:
        diag = "✅ Good fit"
    print(f"{name:<22} {r['train_acc']:>10.4f} {r['test_acc']:>10.4f} {gap:>10.4f}  {diag}")


## 📊 Step 6B. Model Evaluation — Confusion Matrix & ROC Curves (Best Model)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Model Evaluation — Best Model: {best_model_name}', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No (Pred)', 'Yes (Pred)'],
            yticklabels=['No (Actual)', 'Yes (Actual)'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# ROC Curves — all models
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Guess')
axes[1].set_title('ROC Curves — All Models')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


## 🎯 Step 6C. Feature Importance (Best Model)

In [ ]:
best_m = best['model']
if hasattr(best_m, 'feature_importances_'):
    fi = pd.Series(best_m.feature_importances_, index=X.columns).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    fi.plot(kind='bar', ax=ax, color='#2980b9', edgecolor='black')
    ax.set_title(f'Feature Importance — {best_model_name}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Importance Score')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    print("\n📌 Top 5 most important features:")
    print(fi.head(5))
else:
    print(f"{best_model_name} does not expose feature_importances_ (e.g. Logistic Regression) —"
          " refer to model coefficients instead.")


## 📋 Step 6D. Classification Report (Best Model)

In [ ]:
print(f"Classification Report — {best_model_name}")
print("=" * 55)
print(classification_report(y_test, best['y_pred'], target_names=['No Deposit', 'Yes Deposit']))


## 🆕 Bonus. Predict — New Customer Eligibility
Enter a new customer's details and get an instant prediction on whether they are
**eligible (likely to subscribe)** to the Term Deposit, using the best model found above.


In [ ]:
# ── Re-fit label encoders on full dataset for consistent encoding ────────────
data2 = df.copy()
data2.drop(columns=['duration', 'y'], inplace=True)
data2['was_contacted_before'] = (data2['pdays'] != -1).astype(int)
data2['pdays'] = data2['pdays'].replace(-1, 0)

cat_cols2 = data2.select_dtypes(include='object').columns.tolist()
label_maps = {}
for col in cat_cols2:
    le2 = LabelEncoder()
    le2.fit(data2[col])
    label_maps[col] = le2

# ── New Customer Details (edit these values) ─────────────────────────────
new_customer = {
    'age': 42, 'job': 'management', 'marital': 'married', 'education': 'tertiary',
    'default': 'no', 'balance': 1500, 'housing': 'yes', 'loan': 'no',
    'contact': 'cellular', 'day': 15, 'month': 'may', 'campaign': 2,
    'pdays': -1, 'previous': 0, 'poutcome': 'unknown'
}

new_df = pd.DataFrame([new_customer])
new_df['was_contacted_before'] = (new_df['pdays'] != -1).astype(int)
new_df['pdays'] = new_df['pdays'].replace(-1, 0)

for col in cat_cols2:
    new_df[col] = label_maps[col].transform(new_df[col])

new_df = new_df[selected_features]
new_df_scaled = scaler_adv.transform(new_df)

pred = best_m.predict(new_df_scaled)[0]
proba = best_m.predict_proba(new_df_scaled)[0][1]

print(f"Prediction : {'✅ Likely to Subscribe (YES)' if pred == 1 else '❌ Unlikely to Subscribe (NO)'}")
print(f"Confidence : {proba:.2%}")


## 🏁 Summary & Business Insights

| Step | What was done |
|---|---|
| 1. Data Importing | Loaded `bank-full_csv.xls` (45,211 rows, semicolon-delimited) |
| 2. EDA & Correlation | Distribution checks, missing-value/duplicate checks, correlation heatmap |
| 3. Variable Selection | Backward Elimination + Forward Selection (statsmodels p-values/AIC) + RFE, combined into a final feature set |
| 4. Baseline Model | Logistic Regression on selected features |
| 5. Over/Underfitting Check | Train vs test accuracy gap, 5-fold cross-validation, learning curve |
| 6. Advanced Models | Decision Tree, Random Forest, XGBoost (with SMOTE for class imbalance) |

### 📌 Key Insights
1. Variable selection consistently flags **balance, housing, previous outcome, and contact month** as strong predictors.
2. The baseline Logistic Regression provides an interpretable benchmark; tree-based ensembles (Random Forest / XGBoost) generally improve ROC-AUC by capturing non-linear interactions.
3. `duration` was excluded throughout to avoid data leakage, since it is only known after a call takes place.
4. Class imbalance (~7.5:1) was corrected with SMOTE prior to training the advanced models — the baseline model was intentionally left on the untouched split for a fair "as collected" benchmark.
5. Cross-validation standard deviation and the train/test gap are used explicitly to flag any over- or under-fitting before trusting a model's reported score.
